# М01. Самостоятельная лабораторная: какое решение вы готовы защитить?

**Три связанных кейса · постепенное уменьшение подсказок · исходные данные сохраняются.**

После [разбора ESOL](01_guided_case.ipynb) вы умеете отличать измерение от прогноза. Теперь надо проверить другие архивные записи, расследовать несогласованный импорт и выбрать следующий эксперимент для начального досье B (аторвастатин/HMGCR/печёночная экспозиция). Расчёт, который вы сделали в первом кейсе, станет контрольным ориентиром для второго; ограничения обоих понадобятся в третьем.

Числа ESOL не являются данными по печени. Связь кейсов — исследовательский процесс: вопрос → наблюдения → проверка → ограничение → решение. Полные ответы находятся отдельно у преподавателя. Неверный первоначальный прогноз допустим; оценивается, как вы проверили и пересмотрели его.

**Сдача:** notebook с собственным кодом, таблица/график, диагностика импорта, исследовательское решение и новое объяснённое ребро карты. Для этой работы приложения не пишем; будущие CLI/GUI разрабатываются в IDE.


In [1]:
from pathlib import Path
import sys, json, csv, math, statistics, hashlib, importlib.util, os
import matplotlib.pyplot as plt
from IPython.display import display, Image
MODULE = next(candidate for parent in [Path.cwd(), *Path.cwd().parents]
              for candidate in [parent, parent/'course_source/modules/M01']
              if (candidate/'m01.py').is_file())
if str(MODULE) not in sys.path:
    sys.path.insert(0, str(MODULE))
from m01 import META, MEASURED, PREDICTED, locate_snapshot, checksum, format_feedback, metric_invariants, create_dossier
OUT = Path(os.environ.get('M01_WORK_DIR', str(MODULE/'learner_work')))/'independent'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable)
print('RDKit required: no; installed:', importlib.util.find_spec('rdkit') is not None)


Python: /home/miakhlov/course-notebooks/course_source/modules/M01/.venv/bin/python
RDKit required: no; installed: False


## Подготовка: оставшиеся наблюдения

Правило выделения определено заранее. Это новые строки относительно guided, но не новый независимый эксперимент и не независимый от обучения модели test set. Число записей и ID можно проверить, не вычисляя ответы.


In [2]:
from m01 import read_rows,prepare_import_case
source_path=locate_snapshot();source_sha=checksum(source_path)
all_rows=read_rows(source_path)
transfer=[r for r in all_rows if r['row_id']%4==1]
guided_ids={r['row_id'] for r in all_rows if r['row_id']%4==0}
assert len(transfer)==282 and not guided_ids&{r['row_id'] for r in transfer}
display(transfer[:3])


[{'row_id': 1,
  'name': 'Fenfuram',
  'smiles': 'Cc1occc1C(=O)Nc2ccccc2',
  'measured': -3.3,
  'predicted': -2.885},
 {'row_id': 5,
  'name': 'benzothiazole',
  'smiles': 'c2ccc1scnc1c2 ',
  'measured': -1.5,
  'predicted': -2.733},
 {'row_id': 9,
  'name': 'Rotenone',
  'smiles': 'COc5cc4OCC3Oc2c1CC(Oc1ccc2C(=O)C3c4cc5OC)C(C)=C ',
  'measured': -4.42,
  'predicted': -5.246}]

## Кейс 1 — перенос знакомого метода: тот же вопрос, другие записи

**Научная ситуация.** В разобранной части средняя ошибка скрыла сильное расхождение отдельной записи. Перед использованием того же способа анализа на другой части архива проверьте, сохраняется ли эта картина. Нужен вывод о риске отдельных прогнозов, а не попытка добиться одинаковой MAE двух частей.

**Выбор до кода:** какую колонку считаете наблюдением, какую прогнозом? Какое сравнение будет отвечать на вопрос? Что могло бы опровергнуть ваше ожидание?

Вход — `transfer`, 282 подлинные строки. Рассчитайте signed residuals, MAE/RMSE, постройте parity-график с единицами, найдите наибольшее расхождение и переведите его в кратность концентраций. Сравните вывод с guided, сохранив ID и все записи. Не подбирайте subset по ошибке.

**Артефакт:** CSV `row_id, name, measured, predicted, residual`, график и 3–5 предложений «наблюдение → ограничение → рекомендация». В `answer_1` верните `row_ids`, `MAE_logS`, `RMSE_logS`, `largest_row_id`, `conclusion`; поля сами по себе не являются правильным ответом.

<details><summary>Подсказка 1 — как поставить сравнение</summary>

Вычислите разность в одной шкале для каждой пары. Сначала убедитесь, что строкам соответствуют те же row_id. Повторять fitting модели не требуется.
</details>

<details><summary>Подсказка 2 — если затрудняет математика</summary>

Используйте определения MAE и RMSE из guided. Кратность — 10 в степени модуля логарифмического остатка. Она не равна разности концентраций и не является вероятностью ошибки.
</details>


In [3]:
# Частично подготовлен вход; дальнейший расчёт и график напишите самостоятельно.
errors_c1=None
answer_1=None


In [4]:
print(format_feedback(answer_1,['row_ids','MAE_logS','RMSE_logS','largest_row_id','conclusion']))
if answer_1 is not None and errors_c1 is not None:
    print(metric_invariants(errors_c1,answer_1['MAE_logS'],answer_1['RMSE_logS']))


{'status': 'NOT_RUN', 'scope': 'submission-format only', 'reason': 'Сначала выполните собственное исследование'}


## Кейс 2 — расследование ошибки: прогнозы внезапно стали другими

Вы сохранили ориентир из кейса 1. После импорта **тех же** наблюдений коллега получил совсем другую ошибку и предложил удалить неудобные объекты. Сначала выясните, что изменилось в передаче данных, а не исправляйте модель или измерения.

**Это явно синтетическая неисправность экспортера, внесённая автором курса в производную копию.** Она не приписывается Delaney/DeepChem. Исходные measured и raw CSV не изменены. Числа-основания по-прежнему из реального snapshot.

Вход — `import_case`: таблица с `reported_logS` и метаданные производителя. Выберите: изменился состав записей, шкала, модель или условия измерения? Какие наблюдения различат эти версии? Проверьте хотя бы две гипотезы и укажите, какую отвергли.

Сохраните исходный импорт, предложенное исправление и журнал преобразования с единицами. Пересчитайте метрику; сопоставьте с кейсом 1 по тем же ID. Проверьте SHA256 исходного файла. Не делайте вывод об единицах только из диапазона чисел: logS тоже может быть положительным.

**Артефакт:** диагностическая таблица, исправленный CSV, объяснение причины и проверка восстановления. `answer_2`: `hypothesis`, `corrected_MAE_logS`, `raw_unchanged`, `rejected_hypothesis`, `evidence`. Если установить шкалу нельзя, аргументированное «данных недостаточно» предпочтительнее догадки; для этой специально сконструированной неисправности метаданные предоставлены.

<details><summary>Одна подсказка — где искать причину</summary>

Сравните заголовок столбца с `producer_quantity` и `producer_unit`, затем сопоставьте строки с кейсом 1. Обратное преобразование допустимо только после установления смысла величины.
</details>


In [5]:
import_case=prepare_import_case(transfer)
display(import_case['metadata'])
display(import_case['rows'][:3])
# Собственная диагностика, исправление и проверка; готовой формулы здесь нет.
answer_2=None


{'kind': 'synthetic import fault, not an error attributed to publication',
 'column_header': 'reported_logS',
 'producer_quantity': 'S_predicted',
 'producer_unit': 'mol/L',
 'measurement_unit': 'log10(S / (mol L^-1))',
 'source_sha256': '8c06a76f0c6487d29ab0f903e6a7a7139f189ab3c1178f159c8be8964602f189'}

[{'row_id': 1,
  'name': 'Fenfuram',
  'measured': -3.3,
  'reported_logS': 0.0013031667784523},
 {'row_id': 5,
  'name': 'benzothiazole',
  'measured': -1.5,
  'reported_logS': 0.0018492686189780778},
 {'row_id': 9,
  'name': 'Rotenone',
  'measured': -4.42,
  'reported_logS': 5.675446054085465e-06}]

In [6]:
print(format_feedback(answer_2,['hypothesis','corrected_MAE_logS','raw_unchanged','rejected_hypothesis','evidence']))
assert checksum(source_path)==source_sha


{'status': 'NOT_RUN', 'scope': 'submission-format only', 'reason': 'Сначала выполните собственное исследование'}


## Кейс 3 — самостоятельное исследовательское решение: чего не хватает для печени?

Теперь начните собственное досье B. Для аторвастатина имеется экспериментальная структура комплекса с каталитической частью HMG-CoA-редуктазы (1HWK). Вам предлагают использовать модель водной растворимости как доказательство достаточной внутриклеточной концентрации в печени. Вы уже видели ограничение средней ошибки и влияние неверного импорта; теперь нужно самостоятельно определить, отвечает ли вообще доступная величина на нужный вопрос.

Вход — карточки источников ниже и собственные выводы кейсов 1–2. Измерений печёночной экспозиции, транспорта и конкретной формы вещества в этом пакете нет. Назовите исследуемый объект/форму и условия, сформулируйте опровержимую гипотезу. Разделите опубликованный факт, расчёт и недостающее наблюдение. Выберите **одну следующую проверку** и сравните её с разумной альтернативой: дополнительный QSPR, биохимический assay HMGCR, исследование внутриклеточной экспозиции либо аудит источника. Допустим иной метод с обоснованием.

**Самостоятельная работа:** обоснуйте, какая проверка устранит выбранную неопределённость; задайте измеряемую величину/единицы, контроль, критерий опровержения и ограничения. Не придумывайте концентрацию, IC50, pH или результат опыта. Покажите, как проверки источника и единиц из первых двух кейсов войдут в ваш план.

**Артефакт:** `research_question.md`, запись источников/доказательств и `decisions.md` в собственном досье. Новое ребро `concept_map.json` должно содержать направление, отношение, объяснение, source_id, ограничение и следующую проверку. Не копируйте пример ребра из guided.

`answer_3`: `question`, `hypothesis`, `source_ids`, `method`, `control`, `falsification`, `limitation`, `next_action`. Подсказки алгоритма нет: оценивается аргументация; у разных корректных вопросов могут быть разные следующие эксперименты. При отсутствии достаточных доказательств явно укажите `insufficient_evidence`.


In [7]:
cards=json.loads((MODULE/'data/evidence_cards.json').read_text(encoding='utf-8'))
display(cards)
dossier=create_dossier(OUT/'my_research')
print('Моё досье:',dossier)
answer_3=None


{'version': '2026-11-17',
 'kind': 'authored summaries of checked public sources; not new measurements',
 'cards': [{'id': 'A-LABEL',
   'object': 'istradefylline / A2A',
   'source': 'https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=a7d008cb-b273-4049-a5d2-9c6902910d58',
   'fact': 'Label describes istradefylline as an adenosine A2A receptor antagonist.',
   'does_not_establish': 'BBB, affinity or clinical efficacy of another xanthine; ESOL does not measure these endpoints.'},
  {'id': 'B-STRUCTURE',
   'object': 'atorvastatin / catalytic portion of human HMG-CoA reductase',
   'source': 'https://www.rcsb.org/structure/1HWK',
   'publication': 'https://pubmed.ncbi.nlm.nih.gov/11349148/',
   'fact': '1HWK is an experimental crystal structure of this complex.',
   'does_not_establish': 'Hepatic intracellular exposure, transporter contribution, clinical efficacy of a new molecule or aqueous solubility under unspecified conditions.'},
  {'id': 'B-EXPOSURE-GAP',
   'object': 'atorv

Моё досье: /home/miakhlov/m01-issue4-evidence/linux-20261117-inline/learner_work/independent/my_research


In [8]:
print(format_feedback(answer_3,['question','hypothesis','source_ids','method','control','falsification','limitation','next_action']))
if answer_3 is not None:
    (dossier/'M01_submission.json').write_text(json.dumps(answer_3,ensure_ascii=False,indent=2),encoding='utf-8')


{'status': 'NOT_RUN', 'scope': 'submission-format only', 'reason': 'Сначала выполните собственное исследование'}


## Сдача и обсуждение

Каждый кейс оценивается по 4 критериям (0–1 балл): постановка/источник; метод и артефакт; диагностика/контроль; решение с ограничением. Всего 12. Для кейса 3 расчётное число не требуется, но нужен проверяемый план. Имена файлов, объём текста и наличие ключевых слов сами по себе баллов не дают. Неправильный исходный прогноз не штрафуется, если он честно проверен и пересмотрен.

Самопроверка выше проверяет **формат и внутренние арифметические инварианты**, а не научную правильность, клинический вывод или защиту. Пустое решение остаётся NOT_RUN; перед сдачей преподаватель проверяет исходные данные, фактический код, график, объяснение и связь карты. [Полная рубрика преподавателя](instructor/RUBRIC.md) хранится отдельно от учебных ответов.

Ответьте без кода: что вы выясняли, почему выбрали этот метод, какое неожиданное расхождение проверили, что осталось неизвестно и какой эксперимент нужен дальше? На следующей неделе повторите проверку единиц на новой подписанной величине, затем вернитесь к гипотезе своего досье.
